In [1]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_community.utilities import SQLDatabase
from dataclasses import dataclass
from langchain_core.tools import tool
from langgraph.runtime import  get_runtime
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage

In [2]:
model = ChatOllama(
    model='gemma4:e4b',
)

In [15]:
db = SQLDatabase.from_uri('sqlite:///Chinook.db') # db variable contains the database connection

In [16]:
@dataclass
class RuntimeContext: #  RuntimeContext is a bag that the agent is carrying
    db: SQLDatabase # the database is the item inside the bag


In [17]:
@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results"""
    runtime = get_runtime(RuntimeContext) # get the bag the agent is carrying
    db = runtime.context.db # open the bag and take out the database
    try:
        return db.run(query) # run the SQL query on the database
    except Exception as e:
        return f'Error: {str(e)}'

In [18]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [8]:
agent = create_agent(
    model=model,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

In [9]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (950053cf-1c3e-4d91-8f40-32ce4f0c97e8)
 Call ID: 950053cf-1c3e-4d91-8f40-32ce4f0c97e8
  Args:
    query: SELECT T2.total_amount FROM Customers AS T1 INNER JOIN Invoices AS T2 ON T1.customer_id = T2.customer_id WHERE T1.customer_name = 'Frank Harris' ORDER BY T2.invoice_date DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: Customers
[SQL: SELECT T2.total_amount FROM Customers AS T1 INNER JOIN Invoices AS T2 ON T1.customer_id = T2.customer_id WHERE T1.customer_name = 'Frank Harris' ORDER BY T2.invoice_date DESC LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ====

In [10]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (984aa6af-7d61-4fe6-98a1-065b6dfc08b8)
 Call ID: 984aa6af-7d61-4fe6-98a1-065b6dfc08b8
  Args:
    query: SELECT title FROM results LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: results
[SQL: SELECT title FROM results LIMIT 5;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================
Tool Calls:
  execute_sql (f5360c46-06bc-49e5-885b-ed9d80a21773)
 Call ID: f5360c46-06bc-49e5-885b-ed9d80a21773
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table';
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), 

In [19]:
agent = create_agent(
    model=model,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [20]:
question = 'This is Frank Harris, What was the total on my last invoice?'
steps = []

for step in agent.stream(
        {'messages':[{'role':'user', 'content':question}]},
        {'configurable':{'thread_id': '1'}},
        context=RuntimeContext(db=db),
        stream_mode='values',
):
    step['messages'][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (5e9836b6-ddf6-4abf-bd1c-1a0e94f02f7d)
 Call ID: 5e9836b6-ddf6-4abf-bd1c-1a0e94f02f7d
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table';
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (f2570972-6239-44bf-b353-2aadb6ec3aed)
 Call ID: f2570972-6239-44bf-b353-2aadb6ec3aed
  Args:
    query: SELECT T1.total FROM Invoice AS T1 JOIN Customer AS T2 ON T1.customerId = T2.customerId WHERE T2.name = 'Frank Harris' 

In [22]:
question = 'What is my name? and What were the titles?'
steps = []
for step in agent.stream(
        {'messages':[{'role':'user', 'content':question}]},
        {'configurable':{'thread_id':'1'}},
        context=RuntimeContext(db=db),
        stream_mode='values',
):
    step['messages'][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What is my name? and What were the titles?
================================== Ai Message ==================================

Your name is Frank Harris.

The titles from your last invoice were:
*   Holier Than Thou
*   Through The Never
*   My Friend Of Misery
*   The Wait
*   Blitzkrieg
*   So What
